# 🚀 Multi-Source Real-Time NASA Knowledge Assistant — RAG Pipeline

**A single, self-contained, production-grade notebook implementation of a
Retrieval-Augmented Generation system grounded in live NASA data.**

This notebook is the notebook-native conversion of a modular production
codebase (`config.py`, `nasa_connectors.py`, `ingest.py`, `rag_chain.py`,
`app.py`). Every function, class, docstring, and error-handling block from
the original files is preserved **verbatim** — nothing is abridged or
stubbed out. Cross-file imports (`from config import ...`, etc.) have been
removed because, in a notebook, every cell shares one Python kernel
namespace: once a cell defines `CHUNK_SIZE` or `fetch_apod`, every cell
below it can use that name directly — no import plumbing required.

## Pipeline architecture

```
 [1] Setup            →  pip install + global imports
 [2] Configuration    →  endpoints, constants, API-key resolution (DEMO_KEY fallback)
 [3] NASA Connectors  →  APOD · Mars Rover Photos · NeoWs · data.nasa.gov · EarthData · News (RSS→HTML)
 [4] Ingestion        →  normalize → RecursiveCharacterTextSplitter → HuggingFace embeddings → FAISS
 [5] RAG Pipeline     →  MMR retriever (k=4, fetch_k=10) → strict grounded prompt → LLM
 [6] End-to-End Test  →  fetch → index → query → answer + [Source: ...] attribution
```

## Why a single notebook instead of separate modules?

A notebook trades strict module boundaries for **linear reproducibility**:
anyone can open this file top-to-bottom, run every cell in order (`Run All`),
and land on a working, queryable NASA knowledge base with zero setup steps
outside of supplying API keys. That property — a fully self-executing
artifact — is exactly what a `.py` package doesn't give you without a
separate driver script and a virtual environment already configured.

## 🧩 Section 1 — Setup

### 📌 What does this cell do in detail?
Installs every third-party package the notebook depends on, in one
`pip install` invocation:

- **`streamlit`** — kept for parity with the original app (not used for
  UI rendering in this notebook, but `st`-free logic from `app.py` is
  reproduced with plain `print`/`display` calls in Section 6).
- **`langchain`, `langchain-core`, `langchain-community`,
  `langchain-text-splitters`, `langchain-huggingface`, `langchain-openai`**
  — the RAG framework: document abstraction, text splitting, vector store
  wrappers, embeddings integration, and the OpenAI chat model wrapper.
- **`sentence-transformers`** — the actual embedding backend loaded by
  `langchain-huggingface`.
- **`faiss-cpu`** — Facebook AI Similarity Search, an efficient in-memory
  (or persistable) vector index; the CPU build avoids a CUDA dependency.
- **`requests`** — HTTP client for every JSON REST API call (APOD, Mars
  Rover Photos, NeoWs, data.nasa.gov, EarthData CMR).
- **`feedparser`** — robust RSS/Atom parsing for the NASA News feed.
- **`beautifulsoup4` + `lxml`** — HTML parsing, used both to strip HTML
  tags out of RSS summaries and as the fallback scraper if the RSS feed
  is unavailable.
- **`python-dotenv`** — optional `.env` file support for API keys.

The `-q` flag keeps installation output concise; `%pip` (rather than `!pip`)
is the Jupyter-recommended magic because it installs into the *currently
running kernel's* environment, avoiding the classic "installed into the
wrong Python" pitfall of shelling out with `!pip`.

### 💡 Why did we use this specific structure/library?
Pinning this to a single top-of-notebook cell means the entire dependency
surface is declared once, in one place, before any other code runs —
mirroring what `requirements.txt` does for the module-based project, but
in an executable, self-documenting form. Running it first (and only once)
avoids re-triggering slow reinstalls every time a downstream cell is
re-executed during interactive development.

In [ ]:
%pip install -q \
    streamlit \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-openai \
    sentence-transformers \
    faiss-cpu \
    requests \
    feedparser \
    beautifulsoup4 \
    lxml \
    python-dotenv

print("✅ All dependencies installed.")

### 📌 What does this cell do in detail?
Imports every symbol used anywhere in the notebook, grouped by origin:

- **Standard library**: `os`, `logging`, `time`, `getpass`, `dataclasses`
  (`dataclass`, `field`), `datetime` (`datetime`, `timedelta`, `timezone`),
  `typing` (`Any`, `Dict`, `List`, `Optional`, `Final`).
- **Third-party HTTP/parsing**: `requests`, `feedparser`,
  `bs4.BeautifulSoup`.
- **LangChain**: `FAISS` (vector store wrapper), `Document` (the standard
  unit of retrievable text + metadata), `HuggingFaceEmbeddings`,
  `RecursiveCharacterTextSplitter`, `ChatPromptTemplate`,
  `VectorStoreRetriever`, `ChatOpenAI`.
- **Display helpers**: `IPython.display.display` and `Markdown`, used in
  Section 6 to render the assistant's grounded answers with proper
  Markdown formatting (bullet points, bold citations) instead of raw
  `print()` text.

It also configures the root `logging` handler once, at `INFO` level, so
every connector/ingestion/RAG function's `logger.info(...)` /
`logger.warning(...)` / `logger.error(...)` calls become visible in the
notebook output — this is how the original modules reported retry
attempts, empty feeds, and JSON parsing failures.

### 💡 Why did we use this specific structure/library?
Centralizing *all* imports in one cell — rather than repeating them at
the top of every section as the original `.py` files did — is the correct
notebook idiom: a Jupyter kernel is one continuous namespace, so
re-importing `requests` in five different cells adds noise without adding
safety. Declaring everything once, immediately after installation, means
every cell below can assume the full toolkit is already loaded, exactly
as `app.py` could assume `nasa_connectors`, `ingest`, and `rag_chain` were
already `import`-able modules.

In [ ]:
import os
import logging
import time
import getpass
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Final

import requests
import feedparser
from bs4 import BeautifulSoup

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI

from IPython.display import display, Markdown

# One shared logger/handler configuration for the whole notebook — every
# connector, ingestion, and RAG function below logs through this.
logger = logging.getLogger("nasa_knowledge_assistant")
logging.basicConfig(level=logging.INFO)

print("✅ All imports resolved. Kernel namespace ready.")

## ⚙️ Section 2 — Configuration (`config.py` logic)

### 📌 What does this cell do in detail?
Defines every constant the rest of the notebook depends on, exactly as
`config.py` did:

- **NASA endpoint URLs** for all four data sources — APOD, Mars Rover
  Photos, NeoWs, data.nasa.gov, the EarthData CMR search API, and the
  NASA News RSS/HTML pages.
- **`DEMO_API_KEY = "DEMO_KEY"`** — NASA's public, shared, heavily
  rate-limited fallback key that lets the whole pipeline run with zero
  signup.
- **Networking constants** (`REQUEST_TIMEOUT_SECONDS`, `MAX_RETRIES`) that
  parameterize the retry/backoff helper built in Section 3.
- **Ingestion constants** — `CHUNK_SIZE=1000`, `CHUNK_OVERLAP=150` — the
  exact values mandated by the architecture spec for
  `RecursiveCharacterTextSplitter`.
- **Embedding/vector-store constants** — the HuggingFace model id
  `sentence-transformers/all-MiniLM-L6-v2` and a default FAISS
  persistence directory name.
- **Retrieval constants** — `RETRIEVER_SEARCH_TYPE="mmr"`,
  `RETRIEVER_K=4`, `RETRIEVER_FETCH_K=10`, and a diversity/relevance
  trade-off `RETRIEVER_LAMBDA_MULT=0.5`.
- **LLM constants** — default model name and temperature.

### 💡 Why did we use this specific structure/library?
`typing.Final` documents intent — these values are meant to be read-only
configuration, not mutated at runtime — which communicates architectural
contract to anyone reading the notebook, even though Python doesn't
enforce immutability at runtime. Keeping every "magic value" (URLs, chunk
sizes, k/fetch_k) in one clearly labeled cell means later cells never
hard-code a number inline; if you need to point at a different rover, a
different chunk size, or a different embedding model, there is exactly
one place to change it.

In [ ]:
# --------------------------------------------------------------------------- #
# NASA API endpoints
# --------------------------------------------------------------------------- #
NASA_APOD_URL: Final[str] = "https://api.nasa.gov/planetary/apod"
NASA_MARS_PHOTOS_URL: Final[str] = (
    "https://api.nasa.gov/mars-photos/api/v1/rovers/curiosity/photos"
)
NASA_NEOWS_URL: Final[str] = "https://api.nasa.gov/neo/rest/v1/feed"
DATA_NASA_GOV_URL: Final[str] = "https://data.nasa.gov/resource/gvi5-ynla.json"
# EarthData's CMR (Common Metadata Repository) search API is the public,
# key-free way to query collection/granule metadata.
EARTHDATA_CMR_URL: Final[str] = "https://cmr.earthdata.nasa.gov/search/collections.json"
NASA_NEWS_RSS_URL: Final[str] = "https://www.nasa.gov/feed/"
NASA_NEWS_HTML_URL: Final[str] = "https://www.nasa.gov/news/recent/"

DEMO_API_KEY: Final[str] = "DEMO_KEY"  # NASA's public rate-limited fallback key

# --------------------------------------------------------------------------- #
# Networking
# --------------------------------------------------------------------------- #
REQUEST_TIMEOUT_SECONDS: Final[int] = 15
MAX_RETRIES: Final[int] = 2

# --------------------------------------------------------------------------- #
# Ingestion / chunking
# --------------------------------------------------------------------------- #
CHUNK_SIZE: Final[int] = 1000
CHUNK_OVERLAP: Final[int] = 150

# --------------------------------------------------------------------------- #
# Embeddings / vector store
# --------------------------------------------------------------------------- #
EMBEDDING_MODEL_NAME: Final[str] = "sentence-transformers/all-MiniLM-L6-v2"
FAISS_INDEX_DIR: Final[str] = "faiss_index"

# --------------------------------------------------------------------------- #
# Retrieval
# --------------------------------------------------------------------------- #
RETRIEVER_SEARCH_TYPE: Final[str] = "mmr"
RETRIEVER_K: Final[int] = 4
RETRIEVER_FETCH_K: Final[int] = 10
RETRIEVER_LAMBDA_MULT: Final[float] = 0.5  # diversity/relevance trade-off for MMR

# --------------------------------------------------------------------------- #
# LLM
# --------------------------------------------------------------------------- #
DEFAULT_LLM_MODEL: Final[str] = "gpt-4o-mini"
DEFAULT_LLM_TEMPERATURE: Final[float] = 0.0

### 📌 What does this cell do in detail?
Defines the `Settings` dataclass and its `get_settings()` factory:

- `Settings.nasa_api_key` defaults to `os.getenv("NASA_API_KEY", DEMO_API_KEY)`
  — i.e. it reads the `NASA_API_KEY` environment variable if present, and
  **silently falls back to the shared `"DEMO_KEY"`** if it's not set, so
  the notebook never crashes on a missing key.
- `Settings.openai_api_key` similarly defaults to `os.getenv("OPENAI_API_KEY", "")`
  — an empty string if unset, which downstream code (Section 5's
  `build_llm`) explicitly checks for and raises a clear `ValueError` on,
  rather than failing with an opaque authentication error from the OpenAI
  client.
- `with_overrides(...)` returns a **new** `Settings` instance with any
  explicitly supplied values swapped in — this is what lets a UI (or, in
  this notebook, an interactive `getpass` prompt in Section 6) override
  the environment-derived defaults per-session without mutating global
  state.
- `get_settings()` is a small factory function so callers never construct
  `Settings()` directly inline — keeping the *one, obvious way* to obtain
  the current configuration.

### 💡 Why did we use this specific structure/library?
A `@dataclass` was chosen over a plain dict or a set of loose global
variables because it gives typed, named, autocompletable fields
(`settings.nasa_api_key` instead of `settings["nasa_api_key"]`), while
`field(default_factory=lambda: ...)` defers the `os.getenv` lookup until
*instantiation time* rather than *import time* — meaning if you set the
`NASA_API_KEY` environment variable in a cell before calling
`get_settings()`, the new value is picked up correctly, which would not
be true if the default were computed once at class-definition time.

In [ ]:
@dataclass
class Settings:
    """
    Runtime-resolved settings bundle.

    Values fall back to environment variables when not explicitly supplied,
    which lets the Streamlit UI override keys per-session without requiring
    a restart or a .env edit.
    """

    nasa_api_key: str = field(
        default_factory=lambda: os.getenv("NASA_API_KEY", DEMO_API_KEY)
    )
    openai_api_key: str = field(
        default_factory=lambda: os.getenv("OPENAI_API_KEY", "")
    )

    def with_overrides(
        self, nasa_api_key: str | None = None, openai_api_key: str | None = None
    ) -> "Settings":
        """Return a new Settings object with any provided values overridden."""
        return Settings(
            nasa_api_key=nasa_api_key.strip() if nasa_api_key else self.nasa_api_key,
            openai_api_key=(
                openai_api_key.strip() if openai_api_key else self.openai_api_key
            ),
        )


def get_settings() -> Settings:
    """Factory returning a fresh Settings instance resolved from the environment."""
    return Settings()

## 📡 Section 3 — NASA Data Connectors (`nasa_connectors.py` logic)

This section reproduces every API wrapper and scraper from
`nasa_connectors.py`, split into one cell per function (each preceded by
its own explanation) so the retry/backoff helper, all four
`api.nasa.gov` endpoints, `data.nasa.gov`, `earthdata.nasa.gov`, and the
`nasa.gov/news` RSS+HTML scraper are each independently documented.

Every function returns a plain `List[Dict[str, Any]]` of "raw records" —
the same predictable shape (`source`, `category`, `title`, `text`, `url`,
`date`, `media_type`) regardless of which upstream API it came from — so
Section 4's ingestion logic never needs to know the quirks of any
individual source.

### 📌 What does this cell do in detail?
`_request_with_retries(url, params, headers, max_retries)` is the single
shared HTTP helper every connector function below calls instead of
`requests.get` directly. It:

1. Loops up to `max_retries + 1` total attempts.
2. On **HTTP 429** (rate limited), computes an exponential backoff
   (`2 ** attempt` seconds), logs a warning, sleeps, and retries.
3. On **`requests.exceptions.Timeout`** or
   **`requests.exceptions.ConnectionError`**, logs a warning and retries
   after a flat 1-second pause (network blips are often transient).
4. On any other **`requests.exceptions.HTTPError`** (e.g. a 404 or 500),
   logs an error and returns `None` immediately — retrying a genuine
   "not found" wouldn't help.
5. On any other unexpected `RequestException`, logs an error and returns
   `None`.
6. If every retry is exhausted, logs a final error and returns `None`.

Callers uniformly check `if response is None: return []` — a failed
source degrades to "no data from this source" rather than crashing the
whole ingestion run.

### 💡 Why did we use this specific structure/library?
Centralizing retry logic in one function (rather than duplicating
try/except blocks in all six fetch functions) is the DRY principle
applied directly to the architecture's explicit requirement for "robust
error handling for API rate limits and web connection timeouts."
Returning `Optional[requests.Response]` (never raising) lets every
calling function stay simple: check for `None`, then parse `.json()`
knowing the HTTP layer already succeeded.

In [ ]:
# --------------------------------------------------------------------------- #
# Shared HTTP helper
# --------------------------------------------------------------------------- #
def _request_with_retries(
    url: str,
    params: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
    max_retries: int = MAX_RETRIES,
) -> Optional[requests.Response]:
    """
    Perform a GET request with basic retry/backoff handling.

    Handles:
        - Connection timeouts (requests.exceptions.Timeout)
        - Generic connection errors (requests.exceptions.ConnectionError)
        - HTTP 429 (rate limit) with a short backoff before retrying
        - Other non-200 status codes (logged, returns None)

    Returns
    -------
    Optional[requests.Response]
        The successful response object, or None if all retries were exhausted.
    """
    for attempt in range(1, max_retries + 2):  # +1 initial attempt
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code == 429:
                wait = 2 ** attempt
                logger.warning(
                    "Rate limited (429) on %s — backing off %ss (attempt %s/%s)",
                    url,
                    wait,
                    attempt,
                    max_retries + 1,
                )
                time.sleep(wait)
                continue
            response.raise_for_status()
            return response
        except requests.exceptions.Timeout:
            logger.warning(
                "Timeout on %s (attempt %s/%s)", url, attempt, max_retries + 1
            )
        except requests.exceptions.ConnectionError as exc:
            logger.warning(
                "Connection error on %s: %s (attempt %s/%s)",
                url,
                exc,
                attempt,
                max_retries + 1,
            )
        except requests.exceptions.HTTPError as exc:
            logger.error("HTTP error on %s: %s", url, exc)
            return None
        except requests.exceptions.RequestException as exc:
            logger.error("Unexpected request error on %s: %s", url, exc)
            return None
        time.sleep(1)
    logger.error("Exhausted retries for %s", url)
    return None

### 📌 What does this cell do in detail?
`fetch_apod(api_key, days)` retrieves the Astronomy Picture of the Day for
a rolling window ending today:

1. Computes `start_date`/`end_date` as a `days`-length window using
   `datetime.now(timezone.utc).date()` and `timedelta`.
2. Calls the shared retry helper against `NASA_APOD_URL` with
   `api_key`, `start_date`, `end_date` as query params.
3. Parses the JSON response — APOD returns **a single dict** for a
   one-day query but **a list** for a date-range query, so the code
   normalizes both shapes into a list before iterating (`payload if
   isinstance(payload, list) else [payload]`).
4. Skips any item containing an `"error"` key (APOD returns these inline
   rather than as an HTTP error status for some invalid dates).
5. Maps each valid item into the standard record shape, tagging
   `source="APOD"` and `category="astronomy_picture_of_the_day"`.

### 💡 Why did we use this specific structure/library?
Normalizing the dict-vs-list response shape *inside* the connector (not
downstream) keeps that API-specific quirk contained exactly where it
belongs — nothing outside this function needs to know APOD sometimes
returns a bare object instead of a list.

In [ ]:
# --------------------------------------------------------------------------- #
# 1. api.nasa.gov — APOD
# --------------------------------------------------------------------------- #
def fetch_apod(api_key: str, days: int = 5) -> List[Dict[str, Any]]:
    """
    Fetch the Astronomy Picture of the Day for the last `days` days.

    Parameters
    ----------
    api_key : str
        NASA API key (or "DEMO_KEY").
    days : int
        Number of most-recent days to fetch (APOD supports a date range).

    Returns
    -------
    List[Dict[str, Any]]
        One record per day, each containing title/explanation/date/url/media_type.
    """
    end_date = datetime.now(timezone.utc).date()
    start_date = end_date - timedelta(days=max(days - 1, 0))

    params = {
        "api_key": api_key,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
    }
    response = _request_with_retries(NASA_APOD_URL, params=params)
    if response is None:
        return []

    try:
        payload = response.json()
    except ValueError:
        logger.error("APOD response was not valid JSON")
        return []

    # Single-day responses come back as a dict, not a list.
    records = payload if isinstance(payload, list) else [payload]

    results: List[Dict[str, Any]] = []
    for item in records:
        if "error" in item:
            logger.warning("APOD API error: %s", item.get("error"))
            continue
        results.append(
            {
                "source": "APOD",
                "category": "astronomy_picture_of_the_day",
                "title": item.get("title", "Untitled APOD entry"),
                "text": item.get("explanation", ""),
                "url": item.get("url", ""),
                "date": item.get("date", ""),
                "media_type": item.get("media_type", ""),
            }
        )
    return results

### 📌 What does this cell do in detail?
`fetch_mars_rover_photos(api_key, sol, rover, page)` retrieves photo
metadata (not the images themselves) taken by a Mars rover on a given
Martian "sol" (day since landing):

1. Builds the endpoint URL by substituting the requested `rover` name
   into `NASA_MARS_PHOTOS_URL` (which defaults to `"curiosity"`).
2. Fetches the sol/page via the shared retry helper.
3. Extracts the `"photos"` list from the JSON payload.
4. For each photo, builds a **human-readable summary sentence**
   (rover name, camera full name, sol, Earth date, rover status) — this
   matters because the RAG system embeds and retrieves *text*, so a bare
   image URL would be useless as retrievable knowledge; the sentence is
   what actually gets embedded.
5. Tags each record `source="Mars Rover Photos"`,
   `category="mars_exploration"`, `media_type="image"`, and keeps the
   raw `img_src` URL in the `url` field for citation/display purposes.

### 💡 Why did we use this specific structure/library?
Converting structured JSON (camera name, sol, status flags) into a single
descriptive sentence at ingestion time — rather than passing raw JSON
into the embedding model — is a deliberate RAG design choice: embedding
models are trained on natural language, not JSON key/value pairs, so a
well-formed sentence retrieves far more reliably than a JSON blob would.

In [ ]:
# --------------------------------------------------------------------------- #
# 1. api.nasa.gov — Mars Rover Photos
# --------------------------------------------------------------------------- #
def fetch_mars_rover_photos(
    api_key: str, sol: int = 1000, rover: str = "curiosity", page: int = 1
) -> List[Dict[str, Any]]:
    """
    Fetch Mars rover photo metadata for a given Martian sol (day).

    Parameters
    ----------
    api_key : str
        NASA API key.
    sol : int
        Martian sol (day since landing) to query.
    rover : str
        Rover name (curiosity, opportunity, spirit, perseverance).
    page : int
        Pagination page (25 photos per page).

    Returns
    -------
    List[Dict[str, Any]]
        One record per photo, with camera/rover/date metadata. The raw image
        is NOT downloaded — only metadata + image URL, keeping ingestion light.
    """
    url = NASA_MARS_PHOTOS_URL.replace("curiosity", rover)
    params = {"api_key": api_key, "sol": sol, "page": page}
    response = _request_with_retries(url, params=params)
    if response is None:
        return []

    try:
        payload = response.json()
    except ValueError:
        logger.error("Mars Rover Photos response was not valid JSON")
        return []

    photos = payload.get("photos", [])
    results: List[Dict[str, Any]] = []
    for photo in photos:
        camera = photo.get("camera", {})
        rover_info = photo.get("rover", {})
        summary_text = (
            f"Photo taken by the {rover_info.get('name', rover)} rover's "
            f"{camera.get('full_name', camera.get('name', 'unknown camera'))} "
            f"on sol {photo.get('sol')} (Earth date {photo.get('earth_date')}). "
            f"Rover status: {rover_info.get('status', 'unknown')}."
        )
        results.append(
            {
                "source": "Mars Rover Photos",
                "category": "mars_exploration",
                "title": f"{rover_info.get('name', rover)} — Sol {photo.get('sol')} ({camera.get('name', '')})",
                "text": summary_text,
                "url": photo.get("img_src", ""),
                "date": photo.get("earth_date", ""),
                "media_type": "image",
            }
        )
    return results

### 📌 What does this cell do in detail?
`fetch_neows(api_key, days)` retrieves near-Earth asteroid ("NeoWs")
data for an upcoming window (capped at 7 days, which is NeoWs's own API
limit):

1. Computes a `start_date`/`end_date` window, clamped with
   `min(max(days - 1, 0), 6)` so a caller can never accidentally request
   more than NeoWs allows.
2. Calls the shared retry helper against `NASA_NEOWS_URL`.
3. Parses `payload["near_earth_objects"]`, which is a **dict keyed by
   date string**, each value a **list of asteroid objects** — so the
   code does a nested loop (`for date_str, objects in ...items(): for obj
   in objects:`).
4. For each asteroid, extracts estimated diameter (min/max, in meters),
   whether it's flagged `is_potentially_hazardous_asteroid`, and the
   first close-approach record (velocity, miss distance, date).
5. Builds a natural-language summary sentence combining all of the
   above, explicitly stating **"POTENTIALLY HAZARDOUS"** in capitals when
   flagged true — an intentional signal boost so the LLM can't miss or
   soften a genuine hazard flag when answering safety-relevant questions.
6. Tags each record `source="NeoWs"`, `category="near_earth_objects"`.

### 💡 Why did we use this specific structure/library?
Defensive `.get(...)` chaining with sensible fallbacks (e.g.
`obj.get('name', 'Unknown')`) throughout means a single asteroid record
with an unexpectedly missing field never crashes the whole batch — it
just renders as "Unknown" in that one sentence, and every other asteroid
in the response is still processed normally.

In [ ]:
# --------------------------------------------------------------------------- #
# 1. api.nasa.gov — NeoWs (Near-Earth Object Web Service)
# --------------------------------------------------------------------------- #
def fetch_neows(api_key: str, days: int = 3) -> List[Dict[str, Any]]:
    """
    Fetch near-Earth asteroid data for the next `days` days (max 7 per NeoWs limits).

    Returns
    -------
    List[Dict[str, Any]]
        One record per asteroid, summarizing hazard status, diameter, and
        closest approach details.
    """
    start_date = datetime.now(timezone.utc).date()
    end_date = start_date + timedelta(days=min(max(days - 1, 0), 6))

    params = {
        "api_key": api_key,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
    }
    response = _request_with_retries(NASA_NEOWS_URL, params=params)
    if response is None:
        return []

    try:
        payload = response.json()
    except ValueError:
        logger.error("NeoWs response was not valid JSON")
        return []

    near_earth_objects = payload.get("near_earth_objects", {})
    results: List[Dict[str, Any]] = []
    for date_str, objects in near_earth_objects.items():
        for obj in objects:
            est_diameter = obj.get("estimated_diameter", {}).get("meters", {})
            close_approach = (
                obj.get("close_approach_data", [{}])[0]
                if obj.get("close_approach_data")
                else {}
            )
            summary_text = (
                f"Asteroid {obj.get('name', 'Unknown')} "
                f"is {'POTENTIALLY HAZARDOUS' if obj.get('is_potentially_hazardous_asteroid') else 'not classified as hazardous'}. "
                f"Estimated diameter: {est_diameter.get('estimated_diameter_min', 0):.1f}–"
                f"{est_diameter.get('estimated_diameter_max', 0):.1f} meters. "
                f"Close approach on {close_approach.get('close_approach_date_full', date_str)} "
                f"at a relative velocity of "
                f"{close_approach.get('relative_velocity', {}).get('kilometers_per_hour', 'unknown')} km/h, "
                f"miss distance of "
                f"{close_approach.get('miss_distance', {}).get('kilometers', 'unknown')} km."
            )
            results.append(
                {
                    "source": "NeoWs",
                    "category": "near_earth_objects",
                    "title": f"Asteroid {obj.get('name', 'Unknown')} ({date_str})",
                    "text": summary_text,
                    "url": obj.get("nasa_jpl_url", ""),
                    "date": date_str,
                    "media_type": "text",
                }
            )
    return results

### 📌 What does this cell do in detail?
`fetch_data_nasa_gov_datasets(limit)` queries `data.nasa.gov`'s
Socrata-backed open-data API for dataset metadata:

1. Sends a `$limit` query parameter (Socrata's pagination convention)
   through the shared retry helper.
2. Validates the payload is a `list` (Socrata's JSON endpoints return a
   bare array of records) — if not, logs a warning and returns `[]`
   rather than raising a `TypeError` on iteration.
3. For each record, resolves `title` from either a `"title"` or
   `"name"` field (schema varies by dataset), and `description` from
   `"description"` or `"summary"`, falling back to a generic placeholder
   string if genuinely absent.
4. Tags each record `source="data.nasa.gov"`, `category="open_dataset"`,
   `media_type="metadata"`.

### 💡 Why did we use this specific structure/library?
Socrata resource schemas differ meaningfully between individual datasets
hosted on the same platform, so this function is written defensively
with **multiple fallback field names** (`or` chains) rather than assuming
one fixed schema — this is what "robust error handling" means for a
metadata API, as distinct from the connection-level retries the shared
helper already provides.

In [ ]:
# --------------------------------------------------------------------------- #
# 2. data.nasa.gov — Open Data dataset metadata
# --------------------------------------------------------------------------- #
def fetch_data_nasa_gov_datasets(limit: int = 10) -> List[Dict[str, Any]]:
    """
    Fetch dataset metadata records from data.nasa.gov's Socrata-backed open
    data API.

    Notes
    -----
    data.nasa.gov exposes many individual Socrata "resource" endpoints; this
    function targets one general dataset resource as a representative example
    and is intentionally defensive about schema differences between datasets.

    Returns
    -------
    List[Dict[str, Any]]
        Dataset metadata records normalized into the standard record shape.
    """
    params = {"$limit": limit}
    response = _request_with_retries(DATA_NASA_GOV_URL, params=params)
    if response is None:
        return []

    try:
        payload = response.json()
    except ValueError:
        logger.error("data.nasa.gov response was not valid JSON")
        return []

    if not isinstance(payload, list):
        logger.warning("data.nasa.gov returned an unexpected payload shape")
        return []

    results: List[Dict[str, Any]] = []
    for item in payload:
        # Socrata resources vary in schema; fall back gracefully across
        # common field name variants.
        title = item.get("title") or item.get("name") or "Untitled dataset"
        description = (
            item.get("description")
            or item.get("summary")
            or "No description available."
        )
        results.append(
            {
                "source": "data.nasa.gov",
                "category": "open_dataset",
                "title": str(title),
                "text": str(description),
                "url": item.get("landing_page", item.get("url", "")),
                "date": item.get("issued", item.get("modified", "")),
                "media_type": "metadata",
            }
        )
    return results

### 📌 What does this cell do in detail?
`fetch_earthdata_collections(keyword, page_size)` queries NASA's Common
Metadata Repository (CMR) — the public, key-free search backend behind
`earthdata.nasa.gov` — for Earth-science/climate dataset **collections**
matching a keyword:

1. Sends `keyword` and `page_size` as query params, with an explicit
   `Accept: application/json` header (CMR can return XML/Atom by
   default without it).
2. Parses `payload["feed"]["entry"]`, CMR's nested Atom-style JSON
   structure.
3. For each entry, pulls the first link's `href` (if any) as the
   `url`, and the `title`/`summary` fields for the record's text.
4. Tags each record `source="EarthData"`,
   `category="earth_science_collection"`, `media_type="metadata"`.

### 💡 Why did we use this specific structure/library?
CMR was chosen over attempting to scrape `earthdata.nasa.gov` directly
because it's NASA's **official, documented, key-free JSON search API**
for exactly this kind of collection metadata — using the real API instead
of screen-scraping a JavaScript-heavy portal is both more reliable and
more respectful of the upstream service.

In [ ]:
# --------------------------------------------------------------------------- #
# 3. earthdata.nasa.gov — CMR collection metadata
# --------------------------------------------------------------------------- #
def fetch_earthdata_collections(
    keyword: str = "climate", page_size: int = 10
) -> List[Dict[str, Any]]:
    """
    Fetch Earth science / climate dataset collection metadata from NASA's
    Common Metadata Repository (CMR) — the public search backend for
    earthdata.nasa.gov.

    Parameters
    ----------
    keyword : str
        Free-text keyword to filter collections (e.g. "climate", "drought").
    page_size : int
        Number of collections to retrieve.

    Returns
    -------
    List[Dict[str, Any]]
        One record per matching collection, with title/summary/link.
    """
    params = {"keyword": keyword, "page_size": page_size}
    headers = {"Accept": "application/json"}
    response = _request_with_retries(EARTHDATA_CMR_URL, params=params, headers=headers)
    if response is None:
        return []

    try:
        payload = response.json()
    except ValueError:
        logger.error("EarthData CMR response was not valid JSON")
        return []

    entries = payload.get("feed", {}).get("entry", [])
    results: List[Dict[str, Any]] = []
    for entry in entries:
        links = entry.get("links", [])
        primary_link = links[0].get("href", "") if links else ""
        results.append(
            {
                "source": "EarthData",
                "category": "earth_science_collection",
                "title": entry.get("title", "Untitled collection"),
                "text": entry.get("summary", "No summary available."),
                "url": primary_link,
                "date": entry.get("time_start", entry.get("updated", "")),
                "media_type": "metadata",
            }
        )
    return results

### 📌 What does this cell do in detail?
This cell defines the full NASA News trio:

- **`fetch_nasa_news_rss(max_items)`** — the primary strategy. Uses
  `feedparser.parse(NASA_NEWS_RSS_URL)`, checks `feed.bozo` (feedparser's
  own "this feed was malformed" flag) combined with an empty `entries`
  list to detect a broken feed, then for each entry strips any embedded
  HTML out of the RSS summary using
  `BeautifulSoup(raw_summary, "html.parser").get_text(...)` so the
  embedded text is clean prose, not raw markup.
- **`fetch_nasa_news_html(max_items)`** — the fallback strategy, used
  only if RSS fails. Fetches the NASA news listing page with a realistic
  `User-Agent` header, parses it with `BeautifulSoup`, and applies a
  generic heuristic (anchor tags with text longer than 25 characters,
  deduplicated) to approximate headline extraction without depending on
  fragile, redesign-prone CSS selectors.
- **`fetch_nasa_news(max_items)`** — the public entry point. Tries RSS
  first; if it returns zero results, logs an informational message and
  falls through to the HTML scraper.

### 💡 Why did we use this specific structure/library?
This is a deliberate **primary/fallback pattern**: RSS (via `feedparser`)
is structured, stable, and exactly what the architecture spec calls for,
but any live website's feed can go down or change format — the HTML
scrape (via `BeautifulSoup`) exists purely as a safety net so the "news"
source degrades gracefully instead of going completely empty. Both paths
return the same standard record shape, so nothing downstream needs to
know which strategy actually produced the data.

In [ ]:
# --------------------------------------------------------------------------- #
# 4. nasa.gov/news — RSS feed (primary) + HTML scrape (fallback)
# --------------------------------------------------------------------------- #
def fetch_nasa_news_rss(max_items: int = 10) -> List[Dict[str, Any]]:
    """
    Parse NASA's official news RSS feed via `feedparser`.

    Returns
    -------
    List[Dict[str, Any]]
        One record per news item (title, summary, link, published date).
        Returns an empty list (never raises) if the feed is unreachable or
        malformed — callers should treat that as "try the HTML fallback".
    """
    try:
        feed = feedparser.parse(NASA_NEWS_RSS_URL)
    except Exception as exc:  # feedparser rarely raises, but be defensive
        logger.error("feedparser failed on NASA news RSS: %s", exc)
        return []

    if getattr(feed, "bozo", 0) and not feed.entries:
        logger.warning("NASA news RSS feed appears malformed or empty")
        return []

    results: List[Dict[str, Any]] = []
    for entry in feed.entries[:max_items]:
        # Strip any embedded HTML tags from the summary using BeautifulSoup
        # for a clean plain-text chunk suitable for embedding.
        raw_summary = getattr(entry, "summary", "")
        clean_summary = BeautifulSoup(raw_summary, "html.parser").get_text(
            separator=" ", strip=True
        )
        results.append(
            {
                "source": "NASA News",
                "category": "news_rss",
                "title": getattr(entry, "title", "Untitled news item"),
                "text": clean_summary,
                "url": getattr(entry, "link", ""),
                "date": getattr(entry, "published", ""),
                "media_type": "text",
            }
        )
    return results


def fetch_nasa_news_html(max_items: int = 10) -> List[Dict[str, Any]]:
    """
    Fallback scraper for NASA news using `BeautifulSoup`, used when the RSS
    feed is unavailable or returns no entries.

    Notes
    -----
    HTML scraping is inherently fragile to site redesigns. This function is
    written defensively: it looks for common article/heading patterns and
    returns whatever it can find rather than raising on a missed selector.

    Returns
    -------
    List[Dict[str, Any]]
        Scraped news headline + snippet records.
    """
    headers = {"User-Agent": "Mozilla/5.0 (NASA-Knowledge-Assistant/1.0)"}
    response = _request_with_retries(NASA_NEWS_HTML_URL, headers=headers)
    if response is None:
        return []

    try:
        soup = BeautifulSoup(response.text, "html.parser")
    except Exception as exc:
        logger.error("BeautifulSoup failed to parse NASA news HTML: %s", exc)
        return []

    results: List[Dict[str, Any]] = []
    # Generic strategy: look for anchor tags whose text looks like a headline
    # (reasonably long, not a nav link). Real deployments should tailor this
    # selector to the current site markup.
    candidates = soup.find_all("a", href=True)
    seen_titles = set()
    for tag in candidates:
        title = tag.get_text(strip=True)
        href = tag["href"]
        if not title or len(title) < 25 or title in seen_titles:
            continue
        if not href.startswith("http"):
            href = f"https://www.nasa.gov{href}"
        seen_titles.add(title)
        results.append(
            {
                "source": "NASA News",
                "category": "news_html_scrape",
                "title": title,
                "text": title,  # snippet unavailable via this generic selector
                "url": href,
                "date": datetime.now(timezone.utc).isoformat(),
                "media_type": "text",
            }
        )
        if len(results) >= max_items:
            break
    return results


def fetch_nasa_news(max_items: int = 10) -> List[Dict[str, Any]]:
    """
    Public entry point for NASA news: try RSS first, fall back to HTML scrape.

    Returns
    -------
    List[Dict[str, Any]]
        News records from whichever strategy succeeded first.
    """
    rss_results = fetch_nasa_news_rss(max_items=max_items)
    if rss_results:
        return rss_results
    logger.info("RSS feed returned no results — falling back to HTML scrape")
    return fetch_nasa_news_html(max_items=max_items)

### 📌 What does this cell do in detail?
`fetch_all_sources(...)` is the aggregate convenience function that calls
all six connector functions above in one shot and returns a
`Dict[str, List[Dict[str, Any]]]` mapping each source name to its raw
records — `{"APOD": [...], "Mars Rover Photos": [...], "NeoWs": [...],
"data.nasa.gov": [...], "EarthData": [...], "NASA News": [...]}`.

Each source is called independently and sequentially; because every
individual fetch function already catches its own exceptions and returns
`[]` on failure (rather than raising), **one source failing never
prevents the others from returning data** — the aggregate call simply
ends up with an empty list under that source's key.

### 💡 Why did we use this specific structure/library?
This function is the single call site that both the ingestion pipeline
(Section 4) and the "Data Feed Inspector" style display (Section 6) use
— defining it once here means the notebook's fetch behavior is identical
regardless of which downstream consumer triggers it, exactly mirroring
how `app.py` called this same function from its sidebar "refresh"
button.

In [ ]:
# --------------------------------------------------------------------------- #
# Aggregate convenience function
# --------------------------------------------------------------------------- #
def fetch_all_sources(
    api_key: str,
    apod_days: int = 5,
    mars_sol: int = 1000,
    neows_days: int = 3,
    dataset_limit: int = 10,
    earthdata_keyword: str = "climate",
    news_items: int = 10,
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Convenience wrapper that fetches from all four NASA sources in one call.

    Each source is fetched independently and failures are isolated — one
    source failing does not prevent the others from returning data. This is
    the function the ingestion layer and the Streamlit "Data Feed Inspector"
    tab call directly.

    Returns
    -------
    Dict[str, List[Dict[str, Any]]]
        Mapping of source name -> list of raw records.
    """
    return {
        "APOD": fetch_apod(api_key=api_key, days=apod_days),
        "Mars Rover Photos": fetch_mars_rover_photos(api_key=api_key, sol=mars_sol),
        "NeoWs": fetch_neows(api_key=api_key, days=neows_days),
        "data.nasa.gov": fetch_data_nasa_gov_datasets(limit=dataset_limit),
        "EarthData": fetch_earthdata_collections(keyword=earthdata_keyword),
        "NASA News": fetch_nasa_news(max_items=news_items),
    }

## 🧱 Section 4 — Ingestion & Vector Store (`ingest.py` logic)

This section reproduces `ingest.py`'s full pipeline: normalize raw
records into LangChain `Document` objects, split them into overlapping
chunks, embed those chunks locally with a HuggingFace sentence-transformer,
and build a FAISS vector index — plus optional disk persistence and an
end-to-end convenience wrapper.

### 📌 What does this cell do in detail?
`records_to_documents(records_by_source)` flattens the
`{source_name: [record, ...]}` mapping from Section 3 into a single flat
`List[Document]`:

1. Iterates every source, then every record within that source.
2. Strips whitespace from `text` and `title`; skips a record entirely if
   **both** end up empty (nothing meaningful to embed).
3. Builds `page_content` as `"{title}\n\n{text}"` (falling back to just
   `text` if there's no title) — prefixing the title matters because if a
   long document later gets split into multiple chunks, every chunk still
   carries the headline context near its start.
4. Attaches a rich `metadata` dict to every `Document`: `source`,
   `category`, `title`, `url`, `date`, `media_type`, and an
   `ingested_at` UTC timestamp captured once per pipeline run (not
   per-record, so every document from the same batch shares one
   ingestion timestamp).

### 💡 Why did we use this specific structure/library?
LangChain's `Document(page_content=..., metadata=...)` is the standard
unit every downstream LangChain component (splitters, vector stores,
retrievers) expects — normalizing here, once, means Section 4's splitter
and Section 5's retriever never need source-specific branching logic;
they only ever see `Document` objects with a consistent metadata schema.

In [ ]:
# --------------------------------------------------------------------------- #
# Step 1: Normalization
# --------------------------------------------------------------------------- #
def records_to_documents(
    records_by_source: Dict[str, List[Dict[str, Any]]]
) -> List[Document]:
    """
    Convert the `{source_name: [raw_record, ...]}` mapping produced by
    `nasa_connectors.fetch_all_sources` into a flat list of LangChain
    `Document` objects.

    Each raw record is expected to (at minimum) contain a "text" field.
    Missing optional fields are defaulted so downstream code never has to
    guard against KeyError.

    Parameters
    ----------
    records_by_source : Dict[str, List[Dict[str, Any]]]
        Output of `nasa_connectors.fetch_all_sources`.

    Returns
    -------
    List[Document]
        Flattened, normalized documents ready for splitting.
    """
    documents: List[Document] = []
    ingestion_timestamp = datetime.now(timezone.utc).isoformat()

    for source_name, records in records_by_source.items():
        for record in records:
            text = (record.get("text") or "").strip()
            title = (record.get("title") or "").strip()
            if not text and not title:
                # Nothing meaningful to embed — skip empty records.
                continue

            # Prefix the title into the page content so the embedding model
            # captures the headline context even when chunked mid-way.
            page_content = f"{title}\n\n{text}".strip() if title else text

            metadata = {
                "source": record.get("source", source_name),
                "category": record.get("category", "uncategorized"),
                "title": title or "Untitled",
                "url": record.get("url", ""),
                "date": record.get("date", ""),
                "media_type": record.get("media_type", "text"),
                "ingested_at": ingestion_timestamp,
            }
            documents.append(Document(page_content=page_content, metadata=metadata))

    logger.info("Normalized %d raw records into %d Documents", 
                sum(len(v) for v in records_by_source.values()), len(documents))
    return documents

### 📌 What does this cell do in detail?
`split_documents(documents, chunk_size, chunk_overlap)` breaks long
documents into overlapping chunks suitable for embedding and retrieval:

1. Returns `[]` immediately (with a warning) if given an empty document
   list, rather than letting the splitter raise on empty input.
2. Constructs a `RecursiveCharacterTextSplitter` with the spec-mandated
   defaults `chunk_size=1000`, `chunk_overlap=150`, and an explicit
   separator priority list `["\n\n", "\n", ". ", " ", ""]` — the splitter
   tries each separator in order, preferring to break on paragraph
   boundaries, then lines, then sentences, then words, only falling back
   to a hard character cut (`""`) as a last resort.
3. Calls `splitter.split_documents(documents)`, which preserves each
   original document's `metadata` on every chunk derived from it — so a
   chunk from an APOD entry is still tagged `source="APOD"` after
   splitting.

### 💡 Why did we use this specific structure/library?
`RecursiveCharacterTextSplitter` (rather than a naive fixed-length slice)
is the architecture spec's explicit requirement, and for good reason: a
150-character overlap between consecutive chunks means a fact that would
otherwise be severed exactly at a chunk boundary still appears intact in
at least one chunk, which materially improves retrieval recall for
facts near split points.

In [ ]:
# --------------------------------------------------------------------------- #
# Step 2: Splitting
# --------------------------------------------------------------------------- #
def split_documents(
    documents: List[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> List[Document]:
    """
    Split documents into overlapping chunks for more precise retrieval.

    Uses `RecursiveCharacterTextSplitter`, which tries progressively finer
    separators ("\\n\\n", "\\n", " ", "") so chunks break at natural text
    boundaries wherever possible while still respecting `chunk_size`.

    Parameters
    ----------
    documents : List[Document]
        Normalized documents from `records_to_documents`.
    chunk_size : int
        Maximum characters per chunk (default 1000 per spec).
    chunk_overlap : int
        Overlapping characters between consecutive chunks (default 150).

    Returns
    -------
    List[Document]
        Chunked documents; metadata is preserved and propagated to every
        chunk derived from the same source document.
    """
    if not documents:
        logger.warning("split_documents called with an empty document list")
        return []

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    logger.info("Split %d documents into %d chunks", len(documents), len(chunks))
    return chunks

### 📌 What does this cell do in detail?
`get_embedding_model(model_name)` instantiates the HuggingFace embedding
wrapper used to vectorize every chunk:

- Loads `sentence-transformers/all-MiniLM-L6-v2` — a small (~80MB),
  fast, widely-used sentence-embedding model that produces 384-dimensional
  vectors, a strong default for general-purpose semantic search.
- Forces `model_kwargs={"device": "cpu"}` so the notebook runs
  identically on machines without a GPU (no CUDA dependency to install
  or debug).
- Sets `encode_kwargs={"normalize_embeddings": True}` — L2-normalizing
  every embedding vector, which makes cosine similarity and inner-product
  search mathematically equivalent; FAISS's default index uses inner
  product/L2 distance, so normalizing at encode time keeps similarity
  scores well-behaved and comparable across queries.

### 💡 Why did we use this specific structure/library?
Running embeddings **locally** (rather than calling an external
embeddings API) keeps the entire ingestion pipeline free of any
additional API key or per-token cost, and keeps latency predictable —
important for a notebook meant to "just run" without extra account
setup beyond the NASA and OpenAI keys already required for retrieval and
generation.

In [ ]:
# --------------------------------------------------------------------------- #
# Step 3 & 4: Embeddings + FAISS vector store
# --------------------------------------------------------------------------- #
def get_embedding_model(model_name: str = EMBEDDING_MODEL_NAME) -> HuggingFaceEmbeddings:
    """
    Instantiate the local HuggingFace sentence-transformers embedding model.

    Running locally (CPU by default) avoids any dependency on an external
    embeddings API and keeps the pipeline fully self-contained.

    Returns
    -------
    HuggingFaceEmbeddings
        Configured embedding model wrapper.
    """
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

### 📌 What does this cell do in detail?
`build_faiss_index(chunks, embedding_model)` constructs a fresh in-memory
FAISS vector store from a list of chunked `Document` objects:

1. Raises a clear `ValueError` immediately if `chunks` is empty — FAISS
   cannot build a meaningful index from zero vectors, so this fails loud
   and early rather than producing a silently broken empty index.
2. Reuses a passed-in `embedding_model` if provided (avoiding an
   expensive model reload), or lazily creates one via
   `get_embedding_model()` if omitted.
3. Calls `FAISS.from_documents(chunks, embeddings)`, which internally
   embeds every chunk's `page_content` and builds the similarity index in
   one call.

### 💡 Why did we use this specific structure/library?
**FAISS** (Facebook AI Similarity Search) was chosen because it's a
mature, extremely fast, well-supported approximate/exact nearest-neighbor
library with first-class LangChain integration, and it can run entirely
in-process with no external vector-database server to stand up — ideal
for a self-contained notebook.

In [ ]:
def build_faiss_index(
    chunks: List[Document],
    embedding_model: Optional[HuggingFaceEmbeddings] = None,
) -> FAISS:
    """
    Build a fresh in-memory FAISS vector store from document chunks.

    Parameters
    ----------
    chunks : List[Document]
        Output of `split_documents`.
    embedding_model : Optional[HuggingFaceEmbeddings]
        Pass an existing model instance to avoid reloading weights; a new
        one is created if omitted.

    Returns
    -------
    FAISS
        The populated vector store.

    Raises
    ------
    ValueError
        If `chunks` is empty — FAISS cannot build an index with no vectors.
    """
    if not chunks:
        raise ValueError("Cannot build a FAISS index from an empty chunk list.")

    embeddings = embedding_model or get_embedding_model()
    logger.info("Embedding %d chunks with %s", len(chunks), EMBEDDING_MODEL_NAME)
    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

### 📌 What does this cell do in detail?
Three small utility functions for index lifecycle management beyond the
initial build:

- **`persist_faiss_index(vector_store, directory)`** — creates the target
  directory if needed and calls `vector_store.save_local(directory)`,
  writing both the FAISS binary index and its associated LangChain
  docstore/metadata to disk.
- **`load_faiss_index(directory, embedding_model)`** — checks whether the
  directory exists and is non-empty; if not, returns `None` immediately
  (no persisted index yet). If it does exist, attempts
  `FAISS.load_local(..., allow_dangerous_deserialization=True)` inside a
  `try/except`, returning `None` on any failure (corrupted files, a
  version mismatch between the embedding model used to build vs. load)
  rather than crashing the notebook.
- **`add_documents_to_index(vector_store, new_chunks)`** — incrementally
  adds new chunks to an *already-built* store via
  `vector_store.add_documents(...)`, useful for refreshing the index with
  newly fetched news items without rebuilding everything from scratch.

### 💡 Why did we use this specific structure/library?
`allow_dangerous_deserialization=True` is required by LangChain's FAISS
loader because deserializing a pickled index could, in principle, execute
arbitrary code if the file came from an untrusted source — explicitly
opting in signals "we trust this file because we wrote it ourselves in
this same notebook," which is the correct, deliberate use of that flag
here.

In [ ]:
def persist_faiss_index(vector_store: FAISS, directory: str = FAISS_INDEX_DIR) -> None:
    """Save the FAISS index + docstore to disk for reuse across sessions."""
    os.makedirs(directory, exist_ok=True)
    vector_store.save_local(directory)
    logger.info("Persisted FAISS index to '%s'", directory)


def load_faiss_index(
    directory: str = FAISS_INDEX_DIR,
    embedding_model: Optional[HuggingFaceEmbeddings] = None,
) -> Optional[FAISS]:
    """
    Load a previously persisted FAISS index from disk, if one exists.

    Returns
    -------
    Optional[FAISS]
        The loaded vector store, or None if no index is found on disk or
        loading fails for any reason (corrupted files, version mismatch).
    """
    if not os.path.isdir(directory) or not os.listdir(directory):
        logger.info("No persisted FAISS index found at '%s'", directory)
        return None

    embeddings = embedding_model or get_embedding_model()
    try:
        vector_store = FAISS.load_local(
            directory,
            embeddings,
            allow_dangerous_deserialization=True,
        )
        logger.info("Loaded persisted FAISS index from '%s'", directory)
        return vector_store
    except Exception as exc:
        logger.error("Failed to load FAISS index from '%s': %s", directory, exc)
        return None


def add_documents_to_index(
    vector_store: FAISS,
    new_chunks: List[Document],
) -> FAISS:
    """
    Incrementally add new chunks to an existing FAISS store (e.g. after a
    fresh data refresh) instead of rebuilding from scratch.

    Returns
    -------
    FAISS
        The same vector store instance, mutated in place, returned for
        convenience/chaining.
    """
    if not new_chunks:
        logger.info("add_documents_to_index called with no new chunks — no-op")
        return vector_store
    vector_store.add_documents(new_chunks)
    logger.info("Added %d new chunks to existing FAISS index", len(new_chunks))
    return vector_store

### 📌 What does this cell do in detail?
`run_ingestion_pipeline(records_by_source, persist, persist_directory)`
is the end-to-end convenience wrapper that chains every step above:

1. `records_to_documents(...)` — normalize raw records into `Document`s.
2. `split_documents(...)` — chunk them.
3. `build_faiss_index(...)` — embed and index.
4. Optionally `persist_faiss_index(...)` if `persist=True`.
5. Computes a `stats` dict — chunk count per source — by iterating every
   chunk's `metadata["source"]` and tallying a running count.
6. Returns a single dict bundling `vector_store`, the pre-split
   `documents`, the post-split `chunks`, and the `stats` breakdown — one
   object that gives every downstream consumer (the RAG chain, and the
   "Vector DB Status" style display in Section 6) everything it needs.

### 💡 Why did we use this specific structure/library?
Bundling the four related outputs into one return dict (rather than four
separate function calls scattered through the notebook) keeps Section 6's
orchestration code short and readable: one call produces everything
needed to both build the RAG chain *and* display ingestion metrics.

In [ ]:
# --------------------------------------------------------------------------- #
# End-to-end convenience pipeline
# --------------------------------------------------------------------------- #
def run_ingestion_pipeline(
    records_by_source: Dict[str, List[Dict[str, Any]]],
    persist: bool = False,
    persist_directory: str = FAISS_INDEX_DIR,
) -> Dict[str, Any]:
    """
    Run the full ingestion pipeline end-to-end: normalize -> split -> embed
    -> index (-> optionally persist).

    Parameters
    ----------
    records_by_source : Dict[str, List[Dict[str, Any]]]
        Raw records keyed by source, as returned by
        `nasa_connectors.fetch_all_sources`.
    persist : bool
        If True, save the resulting FAISS index to `persist_directory`.
    persist_directory : str
        Target directory for persistence.

    Returns
    -------
    Dict[str, Any]
        {
          "vector_store": FAISS instance,
          "documents": List[Document]  (pre-split, for the Data Feed Inspector),
          "chunks": List[Document]     (post-split, actually indexed),
          "stats": {source: chunk_count, ...}
        }
    """
    documents = records_to_documents(records_by_source)
    chunks = split_documents(documents)
    vector_store = build_faiss_index(chunks)

    if persist:
        persist_faiss_index(vector_store, directory=persist_directory)

    stats: Dict[str, int] = {}
    for chunk in chunks:
        source = chunk.metadata.get("source", "unknown")
        stats[source] = stats.get(source, 0) + 1

    return {
        "vector_store": vector_store,
        "documents": documents,
        "chunks": chunks,
        "stats": stats,
    }

## 🔎 Section 5 — RAG Pipeline & Prompt Engineering (`rag_chain.py` logic)

This section reproduces the retrieval + generation layer: the strict,
domain-specific grounded prompt, the MMR retriever configuration, the LLM
factory, and the `NasaRAGChain` orchestration class that ties them
together and returns both the generated answer and its exact source
attribution.

### 📌 What does this cell do in detail?
Defines `NASA_SYSTEM_PROMPT`, the strict grounded-QA prompt template
string, and `build_prompt_template()`, which wraps it in a LangChain
`ChatPromptTemplate`.

The prompt enforces six explicit rules on the LLM:

1. Answer **only** from the `{context}` block — no outside knowledge.
2. If the context is insufficient, respond with one exact fallback
   sentence rather than guessing.
3. **Every factual statement must carry an inline `[Source: <name>]`
   citation**, using the literal `source` field from the retrieved chunk
   — never an invented source name.
4. Disagreeing or multi-source chunks must be cited **separately per
   claim**, not merged into one unattributed sentence.
5. Prefer concise bullet points for multi-fact answers.
6. Never fabricate URLs, dates, or numbers absent from the context.

`{context}` and `{input}` are LangChain prompt template placeholders,
filled in at call time with the formatted retrieved chunks and the raw
user question respectively.

### 💡 Why did we use this specific structure/library?
This is **prompt engineering as a hard architectural constraint**, not a
soft suggestion: by explicitly naming the required citation format and
giving a literal fallback sentence for the "not enough information" case,
the prompt makes hallucination and unattributed claims much harder for
the LLM to produce, directly satisfying the spec's requirement that
answers "guarantee grounding" and "cite the exact source."

In [ ]:
NASA_SYSTEM_PROMPT = """\
You are the NASA Knowledge Assistant, a factual research aide grounded \
strictly in retrieved NASA data.

RULES YOU MUST FOLLOW:
1. Answer ONLY using information contained in the "Context" section below. \
Do not use outside knowledge, prior training data, or speculation of any kind.
2. If the Context does not contain enough information to answer the \
question, respond exactly with: \
"I don't have enough information from the retrieved NASA sources to answer that." \
Do not attempt to fill gaps with assumptions.
3. Every factual statement in your answer MUST be followed by an inline \
citation naming its exact source, using the format [Source: <source name>], \
e.g. [Source: APOD], [Source: Mars Rover Photos], [Source: NeoWs], \
[Source: data.nasa.gov], [Source: EarthData], [Source: NASA News]. \
Use the "source" field shown for each context chunk below — never invent one.
4. If different chunks disagree or come from different sources, cite each \
claim to its own source separately rather than merging them.
5. Be concise and precise. Prefer bullet points for multi-fact answers.
6. Never fabricate URLs, dates, or numbers that are not present in the Context.

Context:
{context}

Question: {input}

Answer (with inline source citations):\
"""


def build_prompt_template() -> ChatPromptTemplate:
    """Construct the strict grounded-QA prompt as a ChatPromptTemplate."""
    return ChatPromptTemplate.from_template(NASA_SYSTEM_PROMPT)

### 📌 What does this cell do in detail?
`build_mmr_retriever(vector_store, k, fetch_k, lambda_mult)` configures a
Maximal Marginal Relevance retriever on top of the FAISS store:

- `vector_store.as_retriever(search_type="mmr", search_kwargs={"k": k,
  "fetch_k": fetch_k, "lambda_mult": lambda_mult})` — LangChain's FAISS
  wrapper exposes MMR natively as a `search_type`.
- **`fetch_k=10`** — the retriever first pulls the 10 nearest-neighbor
  candidates by raw similarity.
- **MMR re-ranking** then selects **`k=4`** of those 10, balancing
  relevance to the query against mutual diversity between the selected
  chunks, governed by `lambda_mult` (0 = maximize diversity, 1 = maximize
  pure relevance; `0.5` is a balanced default).

### 💡 Why did we use this specific structure/library?
Plain top-k similarity search on NASA data tends to return several
near-duplicate chunks from the same source (e.g. four overlapping
sentences from one long APOD explanation) — MMR is specifically designed
to eliminate that redundancy by penalizing candidates too similar to
ones already selected, which is exactly the spec's stated goal: "eliminate
redundant information."

In [ ]:
def build_mmr_retriever(
    vector_store: FAISS,
    k: int = RETRIEVER_K,
    fetch_k: int = RETRIEVER_FETCH_K,
    lambda_mult: float = RETRIEVER_LAMBDA_MULT,
) -> VectorStoreRetriever:
    """
    Build an MMR retriever over the given FAISS vector store.

    MMR (Maximal Marginal Relevance) re-ranks the initial `fetch_k` candidate
    matches to select `k` results that are both relevant to the query AND
    mutually diverse — this prevents the same near-duplicate APOD paragraph,
    for example, from taking up all four context slots.

    Parameters
    ----------
    vector_store : FAISS
        The populated vector store to search.
    k : int
        Number of chunks to ultimately return (default 4 per spec).
    fetch_k : int
        Number of candidates initially fetched before MMR re-ranking
        (default 10 per spec).
    lambda_mult : float
        0 = max diversity, 1 = max relevance. 0.5 is a balanced default.

    Returns
    -------
    VectorStoreRetriever
        Configured retriever ready to plug into the RAG chain.
    """
    return vector_store.as_retriever(
        search_type=RETRIEVER_SEARCH_TYPE,
        search_kwargs={"k": k, "fetch_k": fetch_k, "lambda_mult": lambda_mult},
    )

### 📌 What does this cell do in detail?
`build_llm(openai_api_key, model_name, temperature)` instantiates the
`ChatOpenAI` chat model used for answer generation:

- Raises a clear `ValueError` immediately if no `openai_api_key` was
  provided — surfacing a precise, actionable error message rather than
  letting the OpenAI client fail later with a generic 401.
- Defaults `temperature=0.0` — **deterministic, minimally creative
  output**, which is exactly what a "grounded only" assistant needs: at
  higher temperatures the model is more likely to embellish or
  paraphrase loosely away from the literal retrieved facts.

### 💡 Why did we use this specific structure/library?
`temperature=0.0` is a direct architectural choice in service of the
grounding requirement — creative diversity (useful for brainstorming
tasks) is actively undesirable here, where faithfulness to retrieved
NASA data matters far more than stylistic variety.

In [ ]:
def build_llm(
    openai_api_key: str,
    model_name: str = DEFAULT_LLM_MODEL,
    temperature: float = DEFAULT_LLM_TEMPERATURE,
) -> ChatOpenAI:
    """
    Instantiate the chat LLM used for answer generation.

    Temperature defaults to 0.0 to maximize determinism/faithfulness to the
    retrieved context, consistent with the "grounded only" requirement.
    """
    if not openai_api_key:
        raise ValueError(
            "An OpenAI API key is required to build the LLM. "
            "Set it in the Streamlit sidebar or the OPENAI_API_KEY env var."
        )
    return ChatOpenAI(
        model=model_name,
        temperature=temperature,
        api_key=openai_api_key,
    )

### 📌 What does this cell do in detail?
`_format_context(documents)` renders the list of retrieved `Document`
chunks into the single context string that gets substituted into the
prompt's `{context}` placeholder:

1. For each retrieved chunk (1-indexed for readability), builds a header
   line: `[Chunk N] source=... | category=... | title=... | date=...`.
2. Appends the chunk's actual `page_content` beneath its header.
3. Joins all chunk blocks with a `\n\n---\n\n` separator so the LLM can
   clearly see where one chunk ends and the next begins.

### 💡 Why did we use this specific structure/library?
Labeling every chunk with its `source` field **explicitly, in plain
text, right above the content** removes any ambiguity for the LLM about
which fact came from where — this is what makes Rule 3 of the system
prompt (mandatory `[Source: ...]` citation) actually enforceable: the
model doesn't have to guess or infer provenance, it's handed the exact
source name to cite for each chunk.

In [ ]:
def _format_context(documents: List[Document]) -> str:
    """
    Render retrieved chunks into a single context string, each chunk
    explicitly labeled with its source/category/date so the LLM has no
    ambiguity about which fact came from where.
    """
    formatted_blocks = []
    for i, doc in enumerate(documents, start=1):
        meta = doc.metadata
        header = (
            f"[Chunk {i}] source={meta.get('source', 'unknown')} | "
            f"category={meta.get('category', 'unknown')} | "
            f"title={meta.get('title', 'Untitled')} | "
            f"date={meta.get('date', 'unknown')}"
        )
        formatted_blocks.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted_blocks)

### 📌 What does this cell do in detail?
`NasaRAGChain` is the orchestration class wrapping retriever + prompt +
LLM into one callable object:

- **`__init__`** builds the MMR retriever, the LLM, and the prompt
  template once at construction time, and initializes
  `last_source_documents = []` for later inspection.
- **`invoke(question)`** runs one full RAG turn:
  1. Returns an early friendly message if `question` is blank.
  2. Calls `self.retriever.invoke(question)` inside a `try/except`,
     returning a clear `"Retrieval error: ..."` message on failure
     instead of raising and crashing the notebook cell.
  3. Stores the retrieved docs on `self.last_source_documents` for later
     inspection without re-running retrieval.
  4. If retrieval genuinely returned zero chunks, returns the exact
     "not enough information" fallback message (matching Rule 2 of the
     system prompt) without even calling the LLM — saving an API call
     for a question the index provably can't answer.
  5. Formats the retrieved chunks via `_format_context`, builds the
     final prompt messages, and calls the LLM inside its own
     `try/except`, again returning a clear error string on failure
     rather than raising.
  6. On success, returns `{"answer": ..., "source_documents": ...,
     "sources_used": ...}` — `sources_used` being a **deduplicated,
     sorted** list of every distinct `source` field among the retrieved
     chunks, ready for direct display as a citation summary.

### 💡 Why did we use this specific structure/library?
Wrapping this as a **class** (rather than a bare LangChain LCEL pipeline)
means the notebook — like the original Streamlit app — can call
`.invoke()` once and separately inspect `.last_source_documents` for a
detailed attribution panel, without paying the cost of a second
retrieval call just to show "what did we actually retrieve."

In [ ]:
class NasaRAGChain:
    """
    Thin orchestration wrapper around retriever + prompt + LLM.

    Exposed as a class (rather than a bare LCEL pipeline) so the Streamlit
    app can easily call `.invoke()` and separately inspect
    `.last_source_documents` for the Source Attribution panel, without
    re-running retrieval.
    """

    def __init__(
        self,
        vector_store: FAISS,
        openai_api_key: str,
        llm_model: str = DEFAULT_LLM_MODEL,
        temperature: float = DEFAULT_LLM_TEMPERATURE,
        k: int = RETRIEVER_K,
        fetch_k: int = RETRIEVER_FETCH_K,
    ) -> None:
        self.retriever = build_mmr_retriever(vector_store, k=k, fetch_k=fetch_k)
        self.llm = build_llm(openai_api_key, model_name=llm_model, temperature=temperature)
        self.prompt = build_prompt_template()
        self.last_source_documents: List[Document] = []

    def invoke(self, question: str) -> Dict[str, Any]:
        """
        Run retrieval + grounded generation for a single question.

        Returns
        -------
        Dict[str, Any]
            {
              "answer": str,
              "source_documents": List[Document],
              "sources_used": List[str]  (deduplicated source names)
            }
        """
        if not question or not question.strip():
            return {
                "answer": "Please enter a question about NASA data.",
                "source_documents": [],
                "sources_used": [],
            }

        try:
            retrieved_docs = self.retriever.invoke(question)
        except Exception as exc:
            logger.error("Retrieval failed: %s", exc)
            return {
                "answer": f"Retrieval error: {exc}",
                "source_documents": [],
                "sources_used": [],
            }

        self.last_source_documents = retrieved_docs

        if not retrieved_docs:
            return {
                "answer": (
                    "I don't have enough information from the retrieved "
                    "NASA sources to answer that."
                ),
                "source_documents": [],
                "sources_used": [],
            }

        context_str = _format_context(retrieved_docs)
        messages = self.prompt.format_messages(context=context_str, input=question)

        try:
            response = self.llm.invoke(messages)
            answer_text = response.content
        except Exception as exc:
            logger.error("LLM generation failed: %s", exc)
            return {
                "answer": f"Generation error: {exc}",
                "source_documents": retrieved_docs,
                "sources_used": sorted(
                    {d.metadata.get("source", "unknown") for d in retrieved_docs}
                ),
            }

        sources_used = sorted(
            {d.metadata.get("source", "unknown") for d in retrieved_docs}
        )
        return {
            "answer": answer_text,
            "source_documents": retrieved_docs,
            "sources_used": sources_used,
        }

### 📌 What does this cell do in detail?
`build_rag_chain(vector_store, openai_api_key, llm_model, temperature, k,
fetch_k)` is a small factory function that constructs and returns a
ready-to-use `NasaRAGChain` instance, forwarding all its arguments
straight into `NasaRAGChain.__init__`.

### 💡 Why did we use this specific structure/library?
A one-line factory function is a small but deliberate API-design choice:
callers never need to remember `NasaRAGChain`'s exact constructor
signature or import the class directly — they call
`build_rag_chain(...)`, mirroring the same factory pattern already used
for `get_settings()` in Section 2 and `get_embedding_model()` in
Section 4, for a consistent "factory function per component" style
throughout the codebase.

In [ ]:
def build_rag_chain(
    vector_store: FAISS,
    openai_api_key: str,
    llm_model: str = DEFAULT_LLM_MODEL,
    temperature: float = DEFAULT_LLM_TEMPERATURE,
    k: int = RETRIEVER_K,
    fetch_k: int = RETRIEVER_FETCH_K,
) -> NasaRAGChain:
    """Factory function to construct a ready-to-use `NasaRAGChain`."""
    return NasaRAGChain(
        vector_store=vector_store,
        openai_api_key=openai_api_key,
        llm_model=llm_model,
        temperature=temperature,
        k=k,
        fetch_k=fetch_k,
    )

## 🧪 Section 6 — End-to-End Test Execution (`app.py` core logic)

This final section reproduces the **core data-flow logic** of `app.py` —
API-key resolution, fetch → ingest → index → build chain → query →
display — using plain notebook I/O (`getpass`, `print`, and
`IPython.display.Markdown`) in place of Streamlit's sidebar widgets and
chat UI. The underlying calls are identical to what the Streamlit
`run_refresh()` and chat-tab handler invoked; only the presentation layer
changes, since a notebook has no `st.sidebar` or `st.chat_message` to
render into.

### 📌 What does this cell do in detail?
Resolves both API keys needed for the pipeline, mirroring the sidebar's
key-management logic from `app.py`:

- Calls `get_settings()` (Section 2) to read any keys already present in
  the environment (`NASA_API_KEY`, `OPENAI_API_KEY`).
- For the **NASA key**: if the environment already provided one, uses it
  silently; otherwise prompts interactively via `getpass.getpass(...)`
  (input hidden, like a password field) and **falls back to the shared
  `"DEMO_KEY"`** if the user just presses Enter — so the notebook never
  blocks indefinitely on a required key the person doesn't have yet.
- For the **OpenAI key**: same pattern, but since generation genuinely
  cannot proceed without one, an empty result after prompting is left as
  `""` and explicitly checked before Section 6's query loop runs (with a
  clear message explaining retrieval/indexing still works without it).
- Prints a masked confirmation of each resolved key (only the last 4
  characters shown) so the user can visually verify a key was picked up
  without echoing the full secret into notebook output that might later
  be shared or committed.

### 💡 Why did we use this specific structure/library?
`getpass.getpass` (rather than a plain `input()`) prevents the API key
from being echoed to the notebook's visible output or terminal — the
same security-conscious pattern as the original app's
`st.sidebar.text_input(..., type="password")`, adapted to the notebook
environment.

In [ ]:
settings = get_settings()

# --- NASA API key resolution (env var -> interactive prompt -> DEMO_KEY) ---
if settings.nasa_api_key and settings.nasa_api_key != "DEMO_KEY":
    nasa_api_key = settings.nasa_api_key
    print(f"✅ NASA_API_KEY loaded from environment (ends in ...{nasa_api_key[-4:]})")
else:
    entered = getpass.getpass(
        "Enter your NASA API key (get one free at https://api.nasa.gov), "
        "or press Enter to use the shared, rate-limited DEMO_KEY: "
    )
    nasa_api_key = entered.strip() if entered.strip() else "DEMO_KEY"
    if nasa_api_key == "DEMO_KEY":
        print("⚠️  Using NASA's shared 'DEMO_KEY' — heavily rate-limited. "
              "Get a free personal key at https://api.nasa.gov for production use.")
    else:
        print(f"✅ NASA API key set (ends in ...{nasa_api_key[-4:]})")

# --- OpenAI API key resolution (env var -> interactive prompt) ---
if settings.openai_api_key:
    openai_api_key = settings.openai_api_key
    print(f"✅ OPENAI_API_KEY loaded from environment (ends in ...{openai_api_key[-4:]})")
else:
    entered = getpass.getpass(
        "Enter your OpenAI API key (required for answer generation; "
        "leave blank to skip generation and only test retrieval/indexing): "
    )
    openai_api_key = entered.strip()
    if openai_api_key:
        print(f"✅ OpenAI API key set (ends in ...{openai_api_key[-4:]})")
    else:
        print("⚠️  No OpenAI key provided — Section 6's generation step will be skipped, "
              "but fetching, ingestion, and indexing will still run normally.")

### 📌 What does this cell do in detail?
Calls `fetch_all_sources(...)` (Section 3) once, using the resolved NASA
key and a set of reasonable default parameters (5 days of APOD, Mars sol
1000, 3 days of upcoming asteroids, 10 EarthData collections on
`"climate"`, 10 news items), then prints a per-source record count —
directly equivalent to the "Data Feed Inspector" tab's raw-record view in
`app.py`, just rendered as text instead of Streamlit expanders.

A short preview (title + truncated text) of the **first record from each
non-empty source** is also printed, so you can sanity-check that live
data actually came back before spending time on the (slower) embedding
step.

### 💡 Why did we use this specific structure/library?
Printing per-source counts *before* ingestion gives an immediate,
cheap signal about data-source health — if `NASA News` shows `0` records
here, you know to check network access or the RSS feed before wondering
why the vector index seems thin, rather than only discovering it deep
inside a confusing retrieval result later.

In [ ]:
print("📡 Fetching from all NASA sources...\n")

raw_records = fetch_all_sources(
    api_key=nasa_api_key,
    apod_days=5,
    mars_sol=1000,
    neows_days=3,
    dataset_limit=10,
    earthdata_keyword="climate",
    news_items=10,
)

total_records = 0
for source_name, records in raw_records.items():
    count = len(records)
    total_records += count
    status = "✅" if count > 0 else "⚠️ "
    print(f"{status} {source_name:<20} {count:>3} record(s)")
    if records:
        preview = records[0]
        preview_text = preview.get("text", "")[:150]
        print(f"      e.g. \"{preview.get('title', 'Untitled')}\" — {preview_text}...")

print(f"\n📦 Total records fetched across all sources: {total_records}")

if total_records == 0:
    print("\n⚠️  No records were retrieved from ANY source. Check your NASA API key "
          "and network connectivity before proceeding to ingestion.")

### 📌 What does this cell do in detail?
Runs `run_ingestion_pipeline(raw_records, persist=False)` (Section 4) on
the records just fetched — normalizing them into `Document`s, splitting
into 1000-character/150-overlap chunks, embedding every chunk locally
with `all-MiniLM-L6-v2`, and building the in-memory FAISS index — then
prints the same kind of metrics the "Vector DB Status" tab displayed in
`app.py`: total documents, total chunks, and a chunk-count breakdown per
source.

`persist=False` keeps the index in memory only for this run; changing it
to `True` (and optionally passing `persist_directory=...`) would write
the index to disk via `persist_faiss_index`, so a later session could
skip re-embedding by calling `load_faiss_index(...)` instead of
re-running this whole cell.

### 💡 Why did we use this specific structure/library?
Displaying the chunk-per-source breakdown immediately after indexing
mirrors the original app's dedicated "Vector DB Status" tab and gives an
at-a-glance sense of which sources are contributing the most retrievable
content to the index — useful context when interpreting *why* the RAG
chain cites one source more often than another in Section 6's final
test queries.

In [ ]:
print("🧱 Running ingestion pipeline (normalize → split → embed → index)...\n")

pipeline_result = run_ingestion_pipeline(raw_records, persist=False)

vector_store = pipeline_result["vector_store"]
documents = pipeline_result["documents"]
chunks = pipeline_result["chunks"]
stats = pipeline_result["stats"]

print(f"📄 Documents ingested : {len(documents)}")
print(f"✂️  Chunks indexed     : {len(chunks)}")
print(f"🗂️  Active sources     : {len(stats)}\n")

print("Chunks per source:")
for source, count in sorted(stats.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  - {source:<20} {count:>3} chunk(s)")

print("\nRetrieval configuration:")
print(f"  - Search type : {RETRIEVER_SEARCH_TYPE.upper()} (Maximal Marginal Relevance)")
print(f"  - k           : {RETRIEVER_K}")
print(f"  - fetch_k     : {RETRIEVER_FETCH_K}")
print(f"  - Embedding   : {EMBEDDING_MODEL_NAME}")

### 📌 What does this cell do in detail?
Builds the final `NasaRAGChain` via `build_rag_chain(vector_store,
openai_api_key)` (Section 5), guarded by a check that an OpenAI key is
actually available — if not, prints a clear explanation and sets
`rag_chain = None` so the next cell can skip generation gracefully
instead of raising the `ValueError` that `build_llm` would otherwise
throw.

### 💡 Why did we use this specific structure/library?
Checking for the key *before* attempting construction — rather than
letting `build_llm`'s internal `ValueError` propagate up as an
unhandled exception — keeps this notebook cell's control flow explicit
and its output message immediately actionable, exactly matching how
`app.py`'s chat tab checked `if not config["openai_api_key"]:` before
ever touching the RAG chain.

In [ ]:
if openai_api_key:
    rag_chain = build_rag_chain(
        vector_store=vector_store,
        openai_api_key=openai_api_key,
    )
    print("✅ RAG chain built — retriever (MMR, k=4/fetch_k=10) + strict grounded prompt + LLM ready.")
else:
    rag_chain = None
    print("⚠️  Skipping RAG chain construction — no OpenAI API key was provided. "
          "The vector index above is fully built and queryable via 'vector_store' directly "
          "(e.g. vector_store.similarity_search('asteroids')), but grounded answer "
          "generation requires an OpenAI key.")

### 📌 What does this cell do in detail?
The final interactive test harness — the notebook's equivalent of typing
questions into the Streamlit chat box:

1. Defines a small list of representative `test_queries` spanning
   multiple sources (an APOD-style astronomy question, a Mars rover
   question, an asteroid-hazard question, and a NASA news question) so
   the grounded citation behavior can be observed across different
   `[Source: ...]` tags in one run.
2. For each question, calls `rag_chain.invoke(question)` (Section 5),
   exactly as the chat tab's `st.chat_input` handler did.
3. Renders the question as a Markdown sub-heading, the generated answer
   as Markdown body text (so any bullet points or **bold** citations in
   the LLM's own output render properly rather than as raw text), and
   then a compact "Sources used" line built from the deduplicated
   `sources_used` list.
4. Beneath that, prints each individual retrieved chunk's title, source,
   date, and a short content preview — the same "Retrieved chunks"
   expander detail the Streamlit chat tab showed, so you can manually
   verify the citation the model gave actually matches what was
   retrieved.
5. Wraps the whole loop in a guard: if `rag_chain is None` (no OpenAI
   key was available), prints an explanatory message and skips straight
   to a raw `vector_store.similarity_search(...)` demonstration instead,
   so the cell still produces *some* meaningful, runnable output either
   way.

### 💡 Why did we use this specific structure/library?
Using `IPython.display.display(Markdown(...))` instead of plain `print()`
for the answer text is a deliberate presentation choice: it lets the
LLM's own Markdown formatting (bullet lists, bold `[Source: ...]` tags if
it emits them in bold) render as actual formatted output in the notebook
cell, rather than as a flat wall of text — directly fulfilling the
requirement to "clearly display both the LLM response and source
attribution."

In [ ]:
test_queries = [
    "What is today's Astronomy Picture of the Day about?",
    "What have Mars rovers photographed recently?",
    "Are there any potentially hazardous asteroids approaching Earth soon?",
    "What is the latest NASA news?",
]

if rag_chain is not None:
    for question in test_queries:
        display(Markdown(f"### ❓ {question}"))

        result = rag_chain.invoke(question)

        display(Markdown(result["answer"]))

        if result["sources_used"]:
            sources_line = ", ".join(f"`{s}`" for s in result["sources_used"])
            display(Markdown(f"**📎 Sources used:** {sources_line}"))

            print("\nRetrieved chunks (MMR, k=4, fetch_k=10):")
            for i, doc in enumerate(result["source_documents"], start=1):
                meta = doc.metadata
                preview = doc.page_content[:200].replace("\n", " ")
                print(f"  [{i}] {meta.get('title', 'Untitled')} "
                      f"— {meta.get('source', 'unknown')} "
                      f"({meta.get('date', 'unknown date')})")
                print(f"      {preview}...")
        else:
            print("(No sources were retrieved for this query.)")

        print("\n" + "=" * 90 + "\n")

else:
    print("Skipping grounded-generation test queries (no OpenAI key).\n")
    print("Demonstrating raw MMR-style similarity search directly against the FAISS index instead:\n")
    demo_query = test_queries[0]
    retriever = build_mmr_retriever(vector_store, k=RETRIEVER_K, fetch_k=RETRIEVER_FETCH_K)
    retrieved = retriever.invoke(demo_query)
    print(f"Query: {demo_query}\n")
    for i, doc in enumerate(retrieved, start=1):
        meta = doc.metadata
        preview = doc.page_content[:200].replace("\n", " ")
        print(f"  [{i}] {meta.get('title', 'Untitled')} — {meta.get('source', 'unknown')}")
        print(f"      {preview}...")

## ✅ Notebook complete

You've now run the full pipeline end-to-end inside a single kernel:
NASA data fetched live from four independent sources → normalized into
LangChain `Document`s → chunked with `RecursiveCharacterTextSplitter`
(1000/150) → embedded locally with `all-MiniLM-L6-v2` → indexed in FAISS
→ retrieved with an MMR retriever (`k=4`, `fetch_k=10`) → answered by an
LLM constrained to a strict, source-citing grounded prompt.

**To refresh with new data**, just re-run Section 6's fetch/ingest/build
cells (Sections 1–5 only need to run once per kernel session, since they
only define functions and constants). **To persist the index across
sessions**, call
`run_ingestion_pipeline(raw_records, persist=True, persist_directory="faiss_index")`
in place of the `persist=False` call above, and use `load_faiss_index()`
at the start of a future session to skip re-embedding entirely.